# Training Dynamics & Loss Spikes

A large training run is mostly uneventful and occasionally catastrophic. Loss spikes,
silent divergence, dead neurons and gradient explosions all announce themselves late — by
which point you may have burned days of cluster time.

The instruments that catch these are cheap and standard: gradient norms, update-to-weight
ratios, activation statistics. What is uncommon is knowing **what a healthy run looks
like**, so that an unhealthy one is recognisable before the loss curve moves.

This notebook builds that baseline and then reproduces the common failures. Fourth topic
in the [RL & Training Dynamics](ppo-from-scratch.ipynb) track; the numerical failure modes
underneath several of these are in
[Mixed-Precision Numerics](../14-pretraining/mixed-precision-numerics.ipynb).

## 1. What & Why

The loss curve is a lagging indicator. By the time it moves, the cause is several hundred
steps behind you. Three quantities move first:

1. **Gradient norm.** Its distribution is remarkably stable in a healthy run. A sudden
   order-of-magnitude jump is the earliest warning available, and it usually precedes a
   loss spike rather than following it.
2. **Update-to-weight ratio** — `‖Δw‖ / ‖w‖` per layer. This is the *effective* learning
   rate, and it is the quantity that should be roughly constant across layers and across
   training. Around `1e-3` is the usual healthy band.
3. **Activation and attention statistics.** Growing activation magnitudes, or attention
   entropy collapsing toward zero, indicate a model heading for numerical trouble well
   before it arrives.

**Why spikes happen at all.** A handful of recurring causes, and they are distinguishable:

- **A pathological batch** — duplicated text, a very long document, corrupted data. The
  spike is sharp, recovery is quick, and it is reproducible by replaying that batch.
- **The learning rate is too high for the current curvature.** Sharp regions of the loss
  landscape appear during training; a step size that was fine yesterday is not today.
- **Numerical overflow** — usually attention logits or the optimiser's second moment. Look
  for `inf` before `NaN`.
- **Optimiser state going stale** after a long plateau, so Adam's second moment
  underestimates the true gradient scale and the first real gradient produces an enormous
  step.

The distinction matters because the fixes differ: skip the batch, lower the LR, widen the
dtype, or reset optimiser state.

## 2. Mental Model

**A vehicle on an unknown road, with the speedometer lagging.**

The loss is your position; the gradient is the road's slope; the learning rate is your
speed. The trouble is that the road's *curvature* changes without warning, and the
appropriate speed depends on curvature, not on how things have gone so far.

- **Warmup** is accelerating gently because you do not yet know the road. At
  initialisation the curvature estimate — Adam's second moment — is based on almost no
  data, so full speed immediately is how runs die in the first hundred steps.
- **Gradient clipping** is a speed limiter. It does not prevent you meeting a cliff; it
  caps how far a single bad reading can throw you. Cheap insurance, and it distorts the
  gradient direction only when it binds.
- **A loss spike** is hitting a bump. Recovering means the suspension absorbed it. Not
  recovering means you have left the road, and because your next gradient is computed
  *from where you now are*, there is no restoring force pulling you back.

That last asymmetry is why training instabilities are dangerous in a way that noisy
evaluation is not: **the process is autoregressive.** A bad step changes where all
subsequent gradients are computed.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Gradient norm** | `‖g‖₂` over all parameters. The single most informative scalar to log. |
| **Gradient clipping** | Rescale `g` when `‖g‖ > c`. Global-norm clipping at `c ≈ 1.0` is near-universal. |
| **Update-to-weight ratio** | `‖Δw‖ / ‖w‖` per layer. Should sit near `1e-3` and be roughly uniform across layers. |
| **Warmup** | Ramping the LR from ~0 over the first few hundred to few thousand steps. |
| **Loss spike** | A sudden large increase. Self-recovering spikes are common; non-recovering ones are fatal. |
| **Divergence** | Loss increasing without recovery, usually ending in `NaN`. |
| **Attention entropy collapse** | Attention distributions becoming near-one-hot; a documented precursor to instability. |
| **Activation growth** | Residual-stream magnitudes growing through training — the route to fp16 overflow. |
| **Skip-and-continue** | On a detected spike, skip the batch and roll back a few steps. Standard practice at scale. |
| **z-loss** | An auxiliary penalty on the softmax normaliser, used to keep logits from drifting large. |
| **Determinism** | Fixed seeds and deterministic kernels, so a failure can be replayed. Worth the throughput cost on big runs. |

## 4. Setup

NumPy. The examples train small models on deliberately awkward problems, because the
failure modes are properties of the optimisation rather than of scale — they are just
cheaper to trigger deliberately.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — what a healthy gradient-norm distribution looks like

You cannot recognise an anomaly without a baseline. Train a small model and record the
distribution.

In [2]:
def make_problem(n=2000, d=40, seed=0):
    r = np.random.default_rng(seed)
    X = r.normal(0, 1, (n, d))
    w = r.normal(0, 1, d)
    y = X @ w + r.normal(0, 0.3, n)
    return X, y, w

X, y, w_star = make_problem()

def train(X, y, lr=0.02, steps=400, clip=None, bad_batch_at=None, warmup=0, seed=0):
    r = np.random.default_rng(seed)
    w = np.random.default_rng(99).normal(0, 0.1, X.shape[1])    # nonzero init
    losses, gnorms, clipped_count = [], [], 0
    for t in range(steps):
        idx = r.integers(0, len(X), 64)
        xb, yb = X[idx], y[idx]
        if bad_batch_at is not None and t == bad_batch_at:
            xb = xb * 50.0                     # a pathological batch: huge inputs
        pred = xb @ w
        err = pred - yb
        g = xb.T @ err / len(yb)
        gn = float(np.linalg.norm(g))
        if clip is not None and gn > clip:
            g = g * (clip / gn)
            clipped_count += 1
        step_lr = lr * min(1.0, (t + 1) / warmup) if warmup else lr
        w -= step_lr * g
        losses.append(float(np.mean((X @ w - y) ** 2)))
        gnorms.append(gn)
    return np.array(losses), np.array(gnorms), clipped_count

losses, gnorms, _ = train(X, y)
print("healthy run, gradient norm distribution:\n")
for label, sl in [("whole run (includes early transient)", gnorms),
                  ("steady state (last 200 steps)", gnorms[200:])]:
    q = np.percentile(sl, [50, 90, 99, 100])
    print(f"{label}")
    print(f"   median {q[0]:7.3f}  p90 {q[1]:7.3f}  p99 {q[2]:7.3f}  max {q[3]:8.3f}"
          f"   max/median {q[3]/q[0]:6.1f}x")
print(f"\nfinal loss {losses[-1]:.4f}  (started {losses[0]:.4f})")
print("\nThe two rows say different things and both matter. Early training has a large")
print("transient -- gradients start big and fall as the model fits -- so any threshold")
print("set on the whole run is meaningless. In STEADY STATE the distribution is tight:")
print("the maximum sits within a small factor of the median. That is the baseline an")
print("alarm should be set against, and it is why the threshold must be a TRAILING")
print("statistic rather than a constant fixed at the start of the run.")
print("\nAn alert on 'gradient norm > 10x the trailing median' costs one line and")
print("catches most of what follows.")

healthy run, gradient norm distribution:

whole run (includes early transient)
   median   0.296  p90   3.693  p99   8.236  max    9.388   max/median   31.8x
steady state (last 200 steps)
   median   0.240  p90   0.288  p99   0.342  max    0.363   max/median    1.5x

final loss 0.0894  (started 41.8744)

The two rows say different things and both matter. Early training has a large
transient -- gradients start big and fall as the model fits -- so any threshold
set on the whole run is meaningless. In STEADY STATE the distribution is tight:
the maximum sits within a small factor of the median. That is the baseline an
alarm should be set against, and it is why the threshold must be a TRAILING
statistic rather than a constant fixed at the start of the run.

An alert on 'gradient norm > 10x the trailing median' costs one line and
catches most of what follows.


### Example 2 — a pathological batch, and what clipping does about it

Inject one bad batch and watch the run with and without gradient clipping.

In [3]:
BAD_AT = 200
for label, clip in [("no clipping", None), ("clip at 10.0", 10.0)]:
    losses, gnorms, nclip = train(X, y, bad_batch_at=BAD_AT, clip=clip)
    print(f"--- {label} ---")
    print(f"  gradient norm at the bad batch : {gnorms[BAD_AT]:.1f}")
    print(f"  loss before / after            : {losses[BAD_AT-1]:.4f} -> {losses[BAD_AT]:.4f}")
    print(f"  loss 50 steps later            : {losses[BAD_AT+50]:.4f}")
    print(f"  final loss                     : {losses[-1]:.4f}")
    print(f"  batches clipped                : {nclip}")
    print()

print("Without clipping, one batch of corrupted inputs produces an enormous gradient")
print("and a single step that undoes a great deal of training. With clipping, the same")
print("batch produces a step no larger than any other, and the run is undisturbed.")
print("Note the count: the threshold binds on ONE batch out of 400 -- the bad one. That")
print("is what a well-chosen clip looks like. Set it well above the steady-state norm")
print("from Example 1 and it is nearly free insurance. Set it near the median and you")
print("are silently rescaling every update, which changes the effective learning rate")
print("and quietly makes your LR schedule a fiction.")

--- no clipping ---
  gradient norm at the bad batch : 20662.6
  loss before / after            : 0.1049 -> 178194.9125
  loss 50 steps later            : 22575.1827
  final loss                     : 62.0436
  batches clipped                : 0

--- clip at 10.0 ---
  gradient norm at the bad batch : 20662.6
  loss before / after            : 0.1049 -> 0.1792
  loss 50 steps later            : 0.1017
  final loss                     : 0.0894
  batches clipped                : 1

Without clipping, one batch of corrupted inputs produces an enormous gradient
and a single step that undoes a great deal of training. With clipping, the same
batch produces a step no larger than any other, and the run is undisturbed.
Note the count: the threshold binds on ONE batch out of 400 -- the bad one. That
is what a well-chosen clip looks like. Set it well above the steady-state norm
from Example 1 and it is nearly free insurance. Set it near the median and you
are silently rescaling every update, which

### Example 3 — warmup, and why the first steps are the dangerous ones

Adaptive optimisers estimate curvature from gradient history. At step 1 there is no
history, so the estimate is poor and the step size is effectively unbounded.

In [4]:
def train_adam(X, y, lr=0.05, steps=300, warmup=0, eps=1e-8, seed=0):
    r = np.random.default_rng(seed)
    w = np.random.default_rng(5).normal(0, 0.1, X.shape[1])    # nonzero init
    m = np.zeros_like(w); v = np.zeros_like(w)
    b1, b2 = 0.9, 0.999
    losses, ratios = [], []
    for t in range(1, steps + 1):
        idx = r.integers(0, len(X), 32)
        g = X[idx].T @ (X[idx] @ w - y[idx]) / len(idx)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        mh = m / (1 - b1 ** t)
        vh = v / (1 - b2 ** t)
        step_lr = lr * min(1.0, t / warmup) if warmup else lr
        update = step_lr * mh / (np.sqrt(vh) + eps)
        w -= update
        losses.append(float(np.mean((X @ w - y) ** 2)))
        ratios.append(float(np.linalg.norm(update) / (np.linalg.norm(w) + 1e-12)))
    return np.array(losses), np.array(ratios)

print(f"{'warmup steps':>13} {'update/weight @step1':>21} {'max update/weight':>19} "
      f"{'final loss':>12}")
for warm in (0, 10, 50, 200):
    losses, ratios = train_adam(X, y, warmup=warm)
    print(f"{warm:13d} {ratios[0]:21.3f} {max(ratios):19.3f} {losses[-1]:12.4f}")

print("\nWith no warmup the very first update is comparable in size to the weights")
print("themselves -- the update-to-weight ratio at step 1 is orders of magnitude above")
print("the healthy ~1e-3 band. On this convex problem the run survives; on a deep")
print("network that step lands somewhere arbitrary and the run may never recover.")
print("\nWarmup exists because Adam's second moment needs a few hundred samples before")
print("it estimates gradient scale usefully. It is not a regularisation trick -- it is")
print("waiting for the optimiser's own statistics to become meaningful.")

 warmup steps  update/weight @step1   max update/weight   final loss
            0                 0.488               0.488       0.0982
           10                 0.056               0.184       0.0972
           50                 0.011               0.084       0.0948
          200                 0.003               0.044       0.0933

With no warmup the very first update is comparable in size to the weights
themselves -- the update-to-weight ratio at step 1 is orders of magnitude above
the healthy ~1e-3 band. On this convex problem the run survives; on a deep
network that step lands somewhere arbitrary and the run may never recover.

Warmup exists because Adam's second moment needs a few hundred samples before
it estimates gradient scale usefully. It is not a regularisation trick -- it is
waiting for the optimiser's own statistics to become meaningful.


### Example 4 — the update-to-weight ratio, layer by layer

The most useful per-layer diagnostic. A layer whose ratio is far from the others is
either not learning or about to destabilise.

In [5]:
# Three "layers" with very different weight scales -- as happens with different
# initialisation schemes, or after some layers have trained more than others.
layers = {"embed": np.full(200, 0.02), "mlp": np.full(200, 0.5), "head": np.full(200, 3.0)}
grads = {"embed": rng.normal(0, 0.01, 200), "mlp": rng.normal(0, 0.01, 200),
         "head": rng.normal(0, 0.01, 200)}

print("with a single global learning rate (plain SGD):\n")
print(f"{'layer':10} {'|w|':>9} {'|grad|':>9} {'|update|':>10} {'update/weight':>15}")
LR = 0.05
for name, w in layers.items():
    upd = LR * grads[name]
    ratio = np.linalg.norm(upd) / np.linalg.norm(w)
    flag = "  <- too small" if ratio < 1e-4 else ("  <- too large" if ratio > 1e-2 else "")
    print(f"{name:10} {np.linalg.norm(w):9.3f} {np.linalg.norm(grads[name]):9.4f} "
          f"{np.linalg.norm(upd):10.5f} {ratio:15.2e}{flag}")

print("\nSame gradient magnitude, same learning rate, and three orders of magnitude")
print("difference in effective step size -- purely because the weight scales differ.")
print("The embedding layer is being trained aggressively while the head barely moves.")
print("\nThis is precisely what adaptive optimisers and per-layer LR scaling exist to")
print("fix, and it is why the ratio -- not the raw gradient norm -- is the quantity to")
print("watch per layer. Healthy is roughly 1e-3, and roughly UNIFORM across layers.")

print("\nthe same layers under Adam-style normalisation:\n")
print(f"{'layer':10} {'update/weight':>15}")
for name, w in layers.items():
    g = grads[name]
    upd = LR * g / (np.sqrt(g ** 2) + 1e-8)          # Adam at steady state: |g|/sqrt(g^2)=1
    print(f"{name:10} {np.linalg.norm(upd)/np.linalg.norm(w):15.2e}")
print("\nAdam removes the gradient-scale dependence but NOT the weight-scale dependence,")
print("which is why decoupled weight decay and careful initialisation still matter.")

with a single global learning rate (plain SGD):

layer            |w|    |grad|   |update|   update/weight
embed          0.283    0.1359    0.00680        2.40e-02  <- too large
mlp            7.071    0.1455    0.00728        1.03e-03
head          42.426    0.1414    0.00707        1.67e-04

Same gradient magnitude, same learning rate, and three orders of magnitude
difference in effective step size -- purely because the weight scales differ.
The embedding layer is being trained aggressively while the head barely moves.

This is precisely what adaptive optimisers and per-layer LR scaling exist to
fix, and it is why the ratio -- not the raw gradient norm -- is the quantity to
watch per layer. Healthy is roughly 1e-3, and roughly UNIFORM across layers.

the same layers under Adam-style normalisation:

layer        update/weight
embed             2.50e+00
mlp               1.00e-01
head              1.67e-02

Adam removes the gradient-scale dependence but NOT the weight-scale dependence

## 6. Gotchas & Pitfalls

- **Logging only the loss.** It is the last thing to move. Log gradient norm, update
  ratios and activation statistics from step 0 — they are nearly free and you cannot add
  them retrospectively to a failed run.
- **Clipping so aggressively that it always binds.** Then you have replaced your optimiser
  with normalised SGD and changed the effective learning rate. Check how often it fires.
- **No warmup with an adaptive optimiser.** Example 3.
- **Assuming a spike that recovered was harmless.** It may have destroyed a
  specialised subnetwork whose loss contribution is small. Compare downstream evaluations,
  not just training loss.
- **Restarting from the crashed checkpoint.** Roll back several hundred steps *before* the
  spike, skip the offending data, and reduce the LR. Restarting at the divergence point
  usually diverges again.
- **Not being able to replay the batch.** Without determinism and data-order logging you
  cannot tell a bad batch from a bad step. On a large run this is worth real throughput.
- **Ignoring slow drifts.** Activation norms creeping up over 100k steps is a genuine
  warning; it ends in fp16 overflow (see
  [Mixed-Precision Numerics](../14-pretraining/mixed-precision-numerics.ipynb)).
- **Blaming the data first.** A spike that reproduces on a *different* batch is not a data
  problem; it is a learning rate or numerics problem.
- **Chasing a spike with a lower LR alone.** If the cause is a pathological batch or
  numerical overflow, the LR is not the fix and you have slowed the run for nothing.

## 7. When to Use vs Alternatives

| Symptom | First move |
|---|---|
| Sharp spike, quick recovery | Log it; check whether the batch is pathological. Often benign |
| Spike with no recovery | Roll back well before it, lower LR, skip the data range |
| `NaN` loss | Find the first `inf` — usually attention logits or optimiser state |
| Divergence in the first 100 steps | Add or lengthen **warmup** |
| Gradient norms creeping up over training | Check activation growth and logit scale; consider **z-loss** |
| One layer's update ratio far from the rest | Initialisation or normalisation problem in that layer |
| Loss flat from step 0 | LR too low, dead activations, or a genuine bug — check the update ratio first |

**The honest position.** Most instability at scale is prevented rather than diagnosed:
warmup, gradient clipping, bf16 instead of fp16, careful initialisation, and z-loss are
cheap and together eliminate the majority of failures. The diagnostics here matter for the
minority that survive all of that.

The habit worth building is **logging the leading indicators before you need them**.
Gradient norm, per-layer update ratio and activation statistics cost almost nothing to
record and are impossible to recover after a run has died. A run that crashes at step
80,000 with only a loss curve is a run you will have to repeat.

## 8. Resources

- [PaLM: Scaling Language Modeling with Pathways](https://arxiv.org/abs/2204.02311) — Section 5.1 documents ~20 loss spikes and the rollback-and-skip mitigation, with evidence it was the data-and-state combination rather than the data alone.
- [OPT-175B logbook](https://github.com/facebookresearch/metaseq/blob/main/projects/OPT/chronicles/OPT175B_Logbook.pdf) — an unusually candid day-by-day record of what actually goes wrong on a large run.
- [Small-scale proxies for large-scale Transformer training instabilities](https://arxiv.org/abs/2309.14322) — reproduces attention-logit growth and output-logit divergence at small scale, and evaluates the standard mitigations.
- [Understanding the Difficulty of Training Transformers](https://arxiv.org/abs/2004.08249) — why warmup is needed and what it is compensating for.
- [On the Variance of the Adaptive Learning Rate and Beyond](https://arxiv.org/abs/1908.03265) — RAdam; the second-moment-variance argument behind warmup.
- [A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/) — Karpathy; the update-to-weight ratio heuristic of Example 4, among much else.
- [Gradient clipping accelerates training](https://arxiv.org/abs/1905.11881) — a theoretical account of why clipping helps beyond simply bounding steps.